In [1]:

import os
import pandas as pd

from config import get_config
from data.loader import load_dataframe, load_data
from preprocessing.filters import apply_filters
from preprocessing.target_transform import TargetTransformer
from preprocessing.split_scale import split_and_scale

from tuning.ann_tune import tune_ann_bayes
from tuning.rf_tune import tune_rf
from tuning.xgb_tune import tune_xgb


In [2]:
df = load_dataframe("hello")  # <-- change path
print("Data loaded:", df.shape)

[INFO] Loaded data: (13134, 30)
Data loaded: (13134, 30)


In [5]:
models = ["ANN"]
targets = ["TAU","H","L"]
stabilities = ["stable", "unstable","all"]

experiments = []

for m in models:
    for t in targets:
        for s in stabilities:
            experiments.append((m, t, s))

print("Total experiments:", len(experiments))
print(experiments)

Total experiments: 9
[('ANN', 'TAU', 'stable'), ('ANN', 'TAU', 'unstable'), ('ANN', 'TAU', 'all'), ('ANN', 'H', 'stable'), ('ANN', 'H', 'unstable'), ('ANN', 'H', 'all'), ('ANN', 'L', 'stable'), ('ANN', 'L', 'unstable'), ('ANN', 'L', 'all')]


In [6]:
# phase1_results = {
#     ('ANN', 'H', 'stable'): {"layer1": 8, "layer2": 0, "layer3": 0, "activation": "relu"},
#     ('ANN', 'H', 'unstable'): {"layer1": 7, "layer2": 0, "layer3": 0, "activation": "relu"},
#     ('ANN', 'TAU', 'stable'): {"layer1": 10, "layer2": 1, "layer3": 0, "activation": "relu"},
#     ('ANN', 'TAU', 'unstable'): {"layer1": 4, "layer2": 1, "layer3": 0, "activation": "tanh"},
# }
phase1_results = {
('ANN','TAU','all'): {"layer1":10,"layer2":0,"layer3":0,"activation":"logistic"},
('ANN','TAU','stable'): {"layer1":6,"layer2":0,"layer3":0,"activation":"tanh"},
('ANN','TAU','unstable'): {"layer1":5,"layer2":4,"layer3":0,"activation":"relu"},
('ANN','H','stable'): {"layer1":8,"layer2":2,"layer3":0,"activation":"relu"},
('ANN','H','unstable'): {"layer1":10,"layer2":0,"layer3":0,"activation":"logistic"},
('ANN','H','all'): {"layer1":6,"layer2":6,"layer3":0,"activation":"relu"},
('ANN','L','stable'): {"layer1":6,"layer2":0,"layer3":0,"activation":"relu"},
('ANN','L','unstable'): {"layer1":6,"layer2":3,"layer3":0,"activation":"relu"},
('ANN','L','all'): {"layer1":6,"layer2":3,"layer3":0,"activation":"relu"},
}

In [7]:
for (model, target, stability), fixed_phase1 in phase1_results.items():
    print("\n" + "="*50)
    print(f"Running: {model} | {target} | {stability}")
    print("="*50)

    # -------- Config --------
    config = get_config(model=model,target=target,stability=stability,mode="tune")
    
    # -------- Data preprocessing --------
    X, y = load_data(df, config)
    X, y = apply_filters(X, y, df, config)
    transformer = TargetTransformer(config)
    y = transformer.transform(y)
    X_train, X_test, y_train, y_test, scaler = split_and_scale(X, y, config)

    # -------- Tuning --------   
    # tune_ann_bayes(X_train,y_train,config,fixed_phase1=None,run_phase1=True,run_phase2=False) #phase1
    tune_ann_bayes(X_train,y_train,config,fixed_phase1=fixed_phase1,run_phase1=False,run_phase2=True) #phase2

    print(f"[DONE] {config.experiment_name}")




Running: ANN | TAU | all
[INFO] Features shape: (13134, 4)
[INFO] Target shape: (13134,)

===== FILTER STATS =====
Total rows: 13134
mask_H         :  13134 
mask_rib       :   6814 
mask_range     :   9799 
mask_nan       :   9918 
mask_qc        :  13134 
------------------------
Final kept     : 6812 
Final removed  : 6322 

[INFO] transforming
[INFO] Train shape: (4768, 4)
[INFO] Test shape: (2044, 4)
[INFO] Using fixed Phase 1 params: {'layer1': 10, 'layer2': 0, 'layer3': 0, 'activation': 'logistic'}
[INFO] Phase 1 best: {'layer1': 10, 'layer2': 0, 'layer3': 0, 'activation': 'logistic'}
[INFO] Running phase2...
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds

/home/balu/.pyenv/versions/3.12.6/envs/py13jpy/lib/python3.12/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.3769423183366303e-08] before, using random point [0.003800318438606909]
  warnings.warn(


Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
[INFO] Phase 2 best: OrderedDict({'alpha': 0.005202149415048875})
[INFO] Final params saved at: saved_models/ANN_TAU_unstable_hyp.json
[DONE] ANN_TAU_unstable

Running: ANN | H | stable
[INFO] Features shape: (13134, 4)
[INFO] Target shape: (13134,)

===== FILTER STATS =====
Total rows: 13134
mask_H         :   4171 
mask_rib       :   6814 
mask_range     :   9626 
mask_nan       :   9801 
mask_qc        :   7583 
------------------------
Final kept     : 1647 
Final removed  : 11487 

[INFO] transforming
[INFO] Train shape: (1152, 4)
[INFO] Test shape: (495, 4)
[INFO] Usi

/home/balu/.pyenv/versions/3.12.6/envs/py13jpy/lib/python3.12/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [0.009999996294184618] before, using random point [0.00014320502822755835]
  warnings.warn(


Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
[INFO] Phase 2 best: OrderedDict({'alpha': 0.00999924451274562})
[INFO] Final params saved at: saved_models/ANN_H_unstable_hyp.json
[DONE] ANN_H_unstable

Running: ANN | H | all
[INFO] Features shape: (13134, 4)
[INFO] Target shape: (13134,)

===== FILTER STATS =====
Total rows: 13134
mask_H         :  13134 
mask_rib       :   6814 
mask_range     :   9626 
mask_nan       :   9801 
mask_qc        :   7583 
------------------------
Final kept     : 4133 
Final removed  : 9001 

[INFO] transforming
[INFO] Train shape: (2893, 4)
[INFO] Test shape: (1240, 4)
[INFO] Using fixed Phase 1 params: {'layer1': 6, 'layer2': 6, 'layer3': 0, 'activation': 'relu'}
[INFO] Phase 1 best: {'layer1': 6, 'layer2': 6, 'layer3': 0, 'activation': 'relu'}
[INFO] Running phase2...
Fitting 5 folds for each of 3 candidates, totalling 1

/home/balu/.pyenv/versions/3.12.6/envs/py13jpy/lib/python3.12/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.3769423183366303e-08] before, using random point [0.003800318438606909]
  warnings.warn(


Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
[INFO] Phase 2 best: OrderedDict({'alpha': 0.008123961759613751})
[INFO] Final params saved at: saved_models/ANN_L_unstable_hyp.json
[DONE] ANN_L_unstable

Running: ANN | L | all
[INFO] Features shape: (13134, 6)
[INFO] Target shape: (13134,)

===== FILTER STATS =====
Total rows: 13134
mask_H         :  13134 
mask_rib       :   6814 
mask_range     :   9827 
mask_nan       :   9900 
mask_qc        :   5625 
------------------------
Final kept     : 3955 
Final removed  : 9179 

[INFO] transforming
[INFO] Train shape: (2768, 6)
[INFO] Test shape: (1187, 6)
[INFO] Using fixe

/home/balu/.pyenv/versions/3.12.6/envs/py13jpy/lib/python3.12/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1e-08] before, using random point [0.005408844386020343]
  warnings.warn(
/home/balu/.pyenv/versions/3.12.6/envs/py13jpy/lib/python3.12/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1e-08] before, using random point [0.0002577805289728385]
  warnings.warn(


Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 3 candidates, totalling 15 fits
[INFO] Phase 2 best: OrderedDict({'alpha': 0.005110448252336984})
[INFO] Final params saved at: saved_models/ANN_L_all_hyp.json
[DONE] ANN_L_all


In [ ]:
models = ["RF","XGB"]
targets = ["H","TAU"]
stabilities = ["stable", "unstable"]

experiments = []

for m in models:
    for t in targets:
        for s in stabilities:
            experiments.append((m, t, s))

print("Total experiments:", len(experiments))
print(experiments)

In [ ]:
for model, target, stability in experiments:
    print("\n" + "="*50)
    print(f"Running: {model} | {target} | {stability}")
    print("="*50)

    # -------- Config --------
    config = get_config(model=model,target=target,stability=stability,mode="tune")
    
    # -------- Data preprocessing --------
    X, y = load_data(df, config)
    X, y = apply_filters(X, y, df, config)
    transformer = TargetTransformer(config)
    y = transformer.transform(y)
    X_train, X_test, y_train, y_test, scaler = split_and_scale(X, y, config)
    # -------- Tuning --------
        
    if model=="RF":
        tune_rf(X_train,y_train,config)
    elif model=="XGB":
        tune_xgb(X_train,y_train,config)
    print(f"[DONE] {config.experiment_name}")